# Hallmark Python Demo

This notebook shows the current Python API with the new config and manifest model.
Each branch has one dataset fmt in `config.yml`, and `data.tsv` stores `sha1` plus fmt parameters.


auto-reload modules when they change


In [ ]:
%load_ext autoreload
%autoreload 2

from hallmark import Repo, ParaFrame


remove any repositories left over from a previous run


In [ ]:
! rm -rf repo*


## 1. Create a Repository


In [ ]:
repo1 = Repo.init("repo1")


repo1/ holds your data files; repo1/.hm/ holds the hallmark index


In [ ]:
! ls -a repo1 repo1/.hm


A fresh repo starts with a commented config template. Use it only if the branch fmt needs regex substitutions.


In [ ]:
! cat repo1/.hm/config.yml


### Bare repository


In [ ]:
repo2 = Repo.init("repo2.hm")


In [ ]:
! ls -a repo2.hm


## 2. Add and Commit Files


In [ ]:
%%bash
pushd repo1
for f in a{0,0.75,0.94}_i{0,30,60,90,120,150,180}.h5; do echo "$f" > "$f"; done
ls *.h5
popd


`repo1.add("a{a}_i{i}.h5")` sets the branch fmt and stages a manifest with `sha1`, `a`, and `i`.


In [ ]:
pf = repo1.add("a{a}_i{i}.h5")
pf


In [ ]:
! echo "=== config.yml ===" && cat repo1/.hm/config.yml && echo "" && echo "=== data.tsv ===" && cat repo1/.hm/data.tsv


In [ ]:
repo1.commit("add spin and inclination files")


## 3. Inspect Status


In [ ]:
repo1.status()


In [ ]:
! cd repo1 && hallmark status


### Config edits

Use `repo1.set_config(...)` when you want to change branch config explicitly, such as switching the branch fmt before `repo1.add(".")`.


In [ ]:
repo1.set_config(remote_name="origin", remote_url="https://example.com/path")
repo1.state.config


## 4. Rebuild the manifest with `repo.add(".")`

Delete some tracked files and rebuild the manifest from current files that still match the branch fmt.


In [ ]:
repo1.checkout("sync-demo")


In [ ]:
%%bash
cd repo1
rm a0.75_i{0,30,60,90,120,150,180}.h5
ls *.h5


In [ ]:
repo1.add(".")


In [ ]:
! cat repo1/.hm/data.tsv


In [ ]:
repo1.commit("sync manifest with dot add")


In [ ]:
repo1.checkout("main")


## 5. Discover and Filter Files with ParaFrame


In [ ]:
pf = ParaFrame.parse("a{a}_i{i}.h5", base_path="repo1")
pf


In [ ]:
ParaFrame.parse("a{a}_i{i}.h5", base_path="repo1", debug=True)


In [ ]:
pf(a=0)


In [ ]:
pf(a=[0, 0.75])


In [ ]:
pf(a=0.94)(i=0)


In [ ]:
pf[pf.i <= 60]


In [ ]:
for path in pf(a=0, i=0).path:
    print(f'Processing "{path}"...')


## 6. Custom Filename Encoding in `config.yml`

For regex substitutions, use the single branch dataset spec under `data:`.
We use a separate repo so the branch still has only one fmt.


In [ ]:
repo_enc = Repo.init("repo_enc")


In [ ]:
%%bash
pushd repo_enc
for f in am0.75_i0.h5 am0.75_i30.h5 am0.75_i60.h5 am0.94_i0.h5 am0.94_i30.h5 am0.94_i60.h5; do echo "$f" > "$f"; done
popd


In [ ]:
%%bash
cat > repo_enc/.hm/config.yml <<'EOF'
data:
  - fmt: "a{aspin}_i{i}.h5"
    encoding:
      aspin: "m([0-9]+(\.[0-9]+)?|\.[0-9]+)"
EOF


In [ ]:
repo_enc.state.config = repo_enc.dothm.load_yml("config")
repo_enc.state.config


In [ ]:
pf_spin = repo_enc.add("a{aspin}_i{i}.h5", encoding=True)
pf_spin


In [ ]:
! echo "=== encoded config.yml ===" && cat repo_enc/.hm/config.yml && echo "" && echo "=== encoded data.tsv ===" && cat repo_enc/.hm/data.tsv


In [ ]:
repo_enc.commit("add negative spin files with encoding")


You can also parse encoded files directly with `ParaFrame.parse()`.


In [ ]:
encodings = repo_enc.state.config.get("data", [])

pf_spin = ParaFrame.parse(
    "a{aspin}_i{i}.h5",
    encodings=encodings,
    base_path="repo_enc",
    encoding=True,
)
pf_spin


In [ ]:
pf_spin(aspin=[-0.75, -0.94])
